# Exercise 2 — Counting Eye Colors

This standalone notebook solves Exercise #2 using modern Python practices.

The goal is to count how many `Person` objects have each eye color listed in a predefined collection, while ensuring that colors with no matching people are still present with a count of `0`.

This solution also adds:
- a clean `Person` data model using `dataclass`;
- a reusable counting function;
- support for any iterable of people;
- deterministic output ordering;
- optional handling of unknown eye colors;
- input validation;
- reproducible sample data;
- edge-case tests;
- a `Counter`-based alternative;
- complexity analysis and implementation notes.

## Problem

Suppose the complete set of supported eye colors is:

In [1]:
eye_colors = (
    "amber",
    "blue",
    "brown",
    "gray",
    "green",
    "hazel",
    "red",
    "violet",
)

We receive a collection of people, each having an `eye_color` attribute.

We want a dictionary containing a count for **every** supported eye color—even if no person has that color.

For example, if no person has `amber` or `blue` eyes, the result must still contain:

```python
{
    "amber": 0,
    "blue": 0,
    ...
}
```

## Data model

The original exercise uses a small mutable class. A frozen `dataclass` is a convenient modern alternative for a simple value object like this.

In [2]:
from collections import Counter
from collections.abc import Iterable, Sequence
from dataclasses import dataclass
from random import Random

In [3]:
@dataclass(frozen=True, slots=True)
class Person:
    """Simple immutable representation of a person for this exercise."""

    eye_color: str

## Reproducible sample data

The original exercise seeds Python's global random-number generator. A local `Random` instance is preferable in reusable code because it avoids changing global random state.

As in the exercise, the generated people are selected only from `eye_colors[2:]`, so nobody should have `amber` or `blue` eyes.

In [4]:
rng = Random(0)

persons = [
    Person(color)
    for color in rng.choices(eye_colors[2:], k=50)
]

persons[:10]

[Person(eye_color='violet'),
 Person(eye_color='red'),
 Person(eye_color='green'),
 Person(eye_color='gray'),
 Person(eye_color='hazel'),
 Person(eye_color='green'),
 Person(eye_color='red'),
 Person(eye_color='gray'),
 Person(eye_color='green'),
 Person(eye_color='hazel')]

## Solution A — Initialize all supported colors first

A direct and readable solution is to create the result dictionary with every valid color initialized to `0`, and then increment counts as people are processed.

This guarantees that colors with no matching people remain present in the output.

In [5]:
def count_eye_colors(
    people: Iterable[Person],
    valid_colors: Sequence[str] = eye_colors,
    *,
    on_unknown: str = "error",
) -> dict[str, int]:
    """Count people by eye color while retaining zero-count colors.

    Parameters
    ----------
    people:
        Any iterable containing objects with an ``eye_color`` attribute.
    valid_colors:
        Ordered sequence of allowed eye colors. Every color appears in the
        returned dictionary, even when its count is zero.
    on_unknown:
        Controls what happens when a person has an eye color that is not
        listed in ``valid_colors``:

        - ``"error"``: raise ``ValueError`` (default).
        - ``"ignore"``: skip that person.

    Returns
    -------
    dict[str, int]
        Mapping of every valid eye color to its count.

    Raises
    ------
    TypeError
        If ``valid_colors`` contains non-string values or an item in
        ``people`` does not expose a string ``eye_color`` attribute.
    ValueError
        If colors are duplicated, ``on_unknown`` is invalid, or an unknown
        eye color is encountered while ``on_unknown='error'``.
    """
    if on_unknown not in {"error", "ignore"}:
        raise ValueError("on_unknown must be either 'error' or 'ignore'.")

    colors = tuple(valid_colors)

    if any(not isinstance(color, str) for color in colors):
        raise TypeError("All valid eye colors must be strings.")

    if len(colors) != len(set(colors)):
        raise ValueError("valid_colors must not contain duplicates.")

    counts = dict.fromkeys(colors, 0)

    for index, person in enumerate(people):
        try:
            color = person.eye_color
        except AttributeError as exc:
            raise TypeError(
                f"Item at position {index} does not have an eye_color attribute."
            ) from exc

        if not isinstance(color, str):
            raise TypeError(
                f"eye_color at position {index} must be a string; "
                f"got {type(color).__name__}."
            )

        if color not in counts:
            if on_unknown == "ignore":
                continue

            raise ValueError(
                f"Unknown eye color {color!r} at position {index}."
            )

        counts[color] += 1

    return counts

In [6]:
counts = count_eye_colors(persons)
counts

{'amber': 0,
 'blue': 0,
 'brown': 3,
 'gray': 10,
 'green': 8,
 'hazel': 7,
 'red': 10,
 'violet': 12}

The important property is immediately visible: `amber` and `blue` are present even though their counts are zero.

In [7]:
assert counts["amber"] == 0
assert counts["blue"] == 0

counts["amber"], counts["blue"]

(0, 0)

## Solution B — Using `Counter`

`Counter` is another natural solution. It counts only values that occur, so we combine it with an initial zero-valued dictionary containing every supported eye color.

This version is particularly concise when the input is already known to be valid.

In [8]:
def count_eye_colors_with_counter(
    people: Iterable[Person],
    valid_colors: Sequence[str] = eye_colors,
    *,
    on_unknown: str = "error",
) -> dict[str, int]:
    """Count eye colors using ``collections.Counter``.

    The returned dictionary preserves the order of ``valid_colors`` and
    contains all valid colors, including those whose count is zero.
    """
    if on_unknown not in {"error", "ignore"}:
        raise ValueError("on_unknown must be either 'error' or 'ignore'.")

    colors = tuple(valid_colors)

    if any(not isinstance(color, str) for color in colors):
        raise TypeError("All valid eye colors must be strings.")

    if len(colors) != len(set(colors)):
        raise ValueError("valid_colors must not contain duplicates.")

    valid_set = set(colors)
    observed: list[str] = []

    for index, person in enumerate(people):
        try:
            color = person.eye_color
        except AttributeError as exc:
            raise TypeError(
                f"Item at position {index} does not have an eye_color attribute."
            ) from exc

        if not isinstance(color, str):
            raise TypeError(
                f"eye_color at position {index} must be a string; "
                f"got {type(color).__name__}."
            )

        if color not in valid_set:
            if on_unknown == "ignore":
                continue

            raise ValueError(
                f"Unknown eye color {color!r} at position {index}."
            )

        observed.append(color)

    frequencies = Counter(observed)

    return {color: frequencies[color] for color in colors}

In [9]:
counter_counts = count_eye_colors_with_counter(persons)
counter_counts

{'amber': 0,
 'blue': 0,
 'brown': 3,
 'gray': 10,
 'green': 8,
 'hazel': 7,
 'red': 10,
 'violet': 12}

## Verify both implementations agree

In [10]:
assert count_eye_colors(persons) == count_eye_colors_with_counter(persons)
print("Both implementations produce the same result.")

Both implementations produce the same result.


## A compact version

If all input data is trusted and validation is unnecessary, the core idea can be expressed very compactly:

In [11]:
def count_eye_colors_compact(
    people: Iterable[Person],
    valid_colors: Sequence[str] = eye_colors,
) -> dict[str, int]:
    """Minimal solution for trusted input."""
    counts = Counter(person.eye_color for person in people)
    return {color: counts[color] for color in valid_colors}

In [12]:
count_eye_colors_compact(persons)

{'amber': 0,
 'blue': 0,
 'brown': 3,
 'gray': 10,
 'green': 8,
 'hazel': 7,
 'red': 10,
 'violet': 12}

## Optional functionality — handling unknown colors

External APIs and databases can contain unexpected values. The main implementation therefore supports two explicit policies:

- `on_unknown="error"` — fail immediately when unexpected data appears;
- `on_unknown="ignore"` — skip unsupported values.

Failing by default is generally safer because silently dropping data can hide upstream data-quality problems.

In [13]:
people_with_unknown_color = [
    Person("brown"),
    Person("green"),
    Person("turquoise"),
]

count_eye_colors(
    people_with_unknown_color,
    on_unknown="ignore",
)

{'amber': 0,
 'blue': 0,
 'brown': 1,
 'gray': 0,
 'green': 1,
 'hazel': 0,
 'red': 0,
 'violet': 0}

## Generator support

The function accepts any iterable rather than requiring a list. This means it also works with generators, which can be useful when processing large datasets lazily.

In [14]:
person_stream = (
    Person(color)
    for color in ["brown", "green", "brown", "hazel"]
)

count_eye_colors(person_stream)

{'amber': 0,
 'blue': 0,
 'brown': 2,
 'gray': 0,
 'green': 1,
 'hazel': 1,
 'red': 0,
 'violet': 0}

## Tests

The following lightweight test suite checks normal behavior as well as important edge cases.

In [15]:
def run_tests() -> None:
    implementations = (
        count_eye_colors,
        count_eye_colors_with_counter,
    )

    expected_sample = {
        "amber": 0,
        "blue": 0,
        "brown": 3,
        "gray": 10,
        "green": 8,
        "hazel": 7,
        "red": 10,
        "violet": 12,
    }

    for implementation in implementations:
        # Reproducible exercise dataset.
        assert implementation(persons) == expected_sample

        # Empty input still returns all possible colors.
        assert implementation([]) == {
            color: 0 for color in eye_colors
        }

        # Basic counting.
        small_sample = [
            Person("brown"),
            Person("brown"),
            Person("green"),
        ]
        result = implementation(small_sample)
        assert result["brown"] == 2
        assert result["green"] == 1
        assert result["amber"] == 0

        # Output order follows valid_colors.
        assert tuple(result) == eye_colors

        # Custom color collection.
        custom = implementation(
            [Person("black"), Person("black"), Person("white")],
            ("black", "white"),
        )
        assert custom == {"black": 2, "white": 1}

        # Generators are supported.
        generated = implementation(
            Person(color)
            for color in ["brown", "green", "brown"]
        )
        assert generated["brown"] == 2
        assert generated["green"] == 1

        # Unknown colors can be ignored explicitly.
        ignored = implementation(
            [Person("brown"), Person("turquoise")],
            on_unknown="ignore",
        )
        assert ignored["brown"] == 1
        assert sum(ignored.values()) == 1

        # Unknown colors raise by default.
        try:
            implementation([Person("turquoise")])
        except ValueError:
            pass
        else:
            raise AssertionError("Unknown colors should raise ValueError")

        # Duplicate valid colors are ambiguous and rejected.
        try:
            implementation([], ("brown", "brown"))
        except ValueError:
            pass
        else:
            raise AssertionError("Duplicate valid colors should raise ValueError")

        # Invalid eye_color values are rejected.
        try:
            implementation([Person(123)])  # type: ignore[arg-type]
        except TypeError:
            pass
        else:
            raise AssertionError("Non-string eye colors should raise TypeError")

    print("All tests passed.")


run_tests()

All tests passed.


## Sanity checks on the generated dataset

The 50 generated people should all belong to the six colors starting at `brown`, and the total of all counts must therefore equal 50.

In [16]:
result = count_eye_colors(persons)

assert len(persons) == 50
assert sum(result.values()) == len(persons)
assert result["amber"] == 0
assert result["blue"] == 0

print(f"People processed: {len(persons)}")
print(f"Total counted:    {sum(result.values())}")

People processed: 50
Total counted:    50


## Complexity analysis

Let:

- `N` = number of people;
- `C` = number of supported eye colors.

For the direct implementation:

- initializing the result dictionary takes **O(C)** time;
- processing all people takes **O(N)** expected time because dictionary membership and updates are expected **O(1)** operations;
- total time complexity is therefore **O(N + C)**;
- additional space is **O(C)**.

The `Counter` implementation has the same asymptotic complexity.

## Recommendation

For this exercise, the most direct solution is:

```python
counts = dict.fromkeys(eye_colors, 0)

for person in persons:
    counts[person.eye_color] += 1
```

It expresses the key requirement very clearly: initialize every possible category first, then count observations.

For production-style code, the validated `count_eye_colors()` function above is preferable because it handles unexpected external data explicitly and accepts arbitrary iterables.